# Simulated Heston Training Data

This notebook creates a compact synthetic training set for the GP emulator. It samples Heston parameter sets, evaluates the FFT pricer on a fixed grid of strikes and maturities, converts prices to Black-Scholes implied volatility, and saves only the model-training columns.

In [1]:
import csv
from pathlib import Path

import numpy as np
from scipy.interpolate import interp1d
from scipy.optimize import brentq
from scipy.stats import norm, qmc


In [2]:
# Market setup used for synthetic option prices.
S0 = 100.0
RISK_FREE_RATE = 0.03
DIVIDEND_YIELD = 0.00

# Latin Hypercube sample size. 1,500 is within the requested 1,000-3,000 range.
N_PARAMETER_SETS = 1_500
RANDOM_SEED = 42

# Seven strikes by five maturities gives 35 prices per parameter set.
STRIKE_GRID = S0 * np.array([0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15])
MATURITY_GRID = np.array([7, 14, 30, 60, 90], dtype=float) / 365.0

# Plausible Heston parameter ranges for SPY-like equity-index surfaces.
PARAMETER_RANGES = {
    "v0": (0.01, 0.25),
    "kappa": (0.10, 5.00),
    "theta": (0.01, 0.25),
    "sigma_v": (0.05, 1.00),
    "rho": (-0.95, 0.00),
}

OUTPUT_PATH = Path("data/simulated_training_data.csv")


## 1. Sample Parameter Sets

Latin Hypercube sampling spreads draws across every parameter range, which helps the emulator see broad coverage without requiring a very large random sample.

In [3]:
def sample_heston_parameters(n_samples, parameter_ranges, seed):
    """Return a list of sampled Heston parameter dictionaries."""
    names = list(parameter_ranges)
    bounds = np.array([parameter_ranges[name] for name in names], dtype=float)

    sampler = qmc.LatinHypercube(d=len(names), seed=seed)
    unit_samples = sampler.random(n=n_samples)
    scaled_samples = qmc.scale(unit_samples, bounds[:, 0], bounds[:, 1])

    return [dict(zip(names, values)) for values in scaled_samples]


parameter_sets = sample_heston_parameters(N_PARAMETER_SETS, PARAMETER_RANGES, RANDOM_SEED)
parameter_sets[:3]


[{'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287},
 {'v0': 0.05848390042373812,
  'kappa': 2.6879802769734984,
  'theta': 0.08507422971115568,
  'sigma_v': 0.1290855280326388,
  'rho': -0.5139185777606672},
 {'v0': 0.04002067231612278,
  'kappa': 4.190105901036428,
  'theta': 0.029576981580787094,
  'sigma_v': 0.9152455843115952,
  'rho': -0.47084749565925726}]

## 2. Price Options and Convert to Implied Volatility

The FFT pricer computes one full strike grid for each maturity. We interpolate those prices to the requested strikes, then invert each call price to implied volatility.

In [4]:
def black_scholes_call(S0, K, T, r, q, sigma):
    """Closed-form Black-Scholes call price."""
    if T <= 0.0:
        return max(S0 - K, 0.0)

    vol_sqrt_T = sigma * np.sqrt(T)
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma**2) * T) / vol_sqrt_T
    d2 = d1 - vol_sqrt_T
    return S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_volatility_call(price, S0, K, T, r, q):
    """Convert a call price to Black-Scholes implied volatility."""
    intrinsic = max(S0 * np.exp(-q * T) - K * np.exp(-r * T), 0.0)
    upper_bound = S0 * np.exp(-q * T)

    if not np.isfinite(price) or price <= intrinsic or price >= upper_bound:
        return np.nan

    objective = lambda sigma: black_scholes_call(S0, K, T, r, q, sigma) - price

    try:
        return brentq(objective, 1e-4, 5.0, maxiter=100)
    except ValueError:
        return np.nan


def heston_log_price_cf(u, S0, T, r, q, v0, kappa, theta, sigma_v, rho):
    """Characteristic function of log(S_T) under the Heston model."""
    u = np.asarray(u, dtype=complex)
    x0 = np.log(S0)
    iu = 1j * u

    a = kappa * theta
    b = kappa - rho * sigma_v * iu
    d = np.sqrt(b**2 + sigma_v**2 * (u**2 + iu))
    g = (b - d) / (b + d)
    exp_dt = np.exp(-d * T)

    # Little Heston Trap form keeps the complex logarithm numerically stable.
    C = iu * (x0 + (r - q) * T) + (a / sigma_v**2) * (
        (b - d) * T - 2.0 * np.log((1.0 - g * exp_dt) / (1.0 - g))
    )
    D = ((b - d) / sigma_v**2) * ((1.0 - exp_dt) / (1.0 - g * exp_dt))
    return np.exp(C + D * v0)


def carr_madan_fft_call_grid(S0, T, r, q, params, alpha=1.5, n_fft=2**11, eta=0.25):
    """Evaluate Heston call prices on a log-strike grid with Carr-Madan FFT."""
    j = np.arange(n_fft)
    v = eta * j
    lam = 2.0 * np.pi / (n_fft * eta)
    b = 0.5 * n_fft * lam
    k = -b + lam * j

    weights = np.ones(n_fft)
    weights[0] = 1.0 / 3.0
    weights[1::2] = 4.0 / 3.0
    weights[2::2] = 2.0 / 3.0

    u = v - 1j * (alpha + 1.0)
    phi = heston_log_price_cf(u, S0, T, r, q, **params)
    denominator = alpha**2 + alpha - v**2 + 1j * (2.0 * alpha + 1.0) * v
    psi = np.exp(-r * T) * phi / denominator

    fft_input = np.exp(1j * b * v) * psi * eta * weights
    fft_values = np.fft.fft(fft_input).real
    strikes = np.exp(k)
    calls = np.exp(-alpha * k) * fft_values / np.pi

    valid = np.isfinite(calls) & (calls > 0.0)
    return strikes[valid], calls[valid]


def carr_madan_fft_call(strikes, S0, T, r, q, params):
    """Interpolate FFT prices to the requested strike grid."""
    grid_strikes, grid_calls = carr_madan_fft_call_grid(S0, T, r, q, params)
    interpolator = interp1d(grid_strikes, grid_calls, kind="cubic", bounds_error=False, fill_value="extrapolate")
    return np.asarray(interpolator(strikes), dtype=float)


In [5]:
def build_training_rows(parameter_sets, strike_grid, maturity_grid, S0, r, q):
    """Build rows with [v0, kappa, theta, sigma_v, rho, K, T, implied_vol]."""
    rows = []

    for params in parameter_sets:
        for T in maturity_grid:
            prices = carr_madan_fft_call(strike_grid, S0, T, r, q, params)

            for K, price in zip(strike_grid, prices):
                implied_vol = implied_volatility_call(price, S0, K, T, r, q)

                if np.isfinite(implied_vol):
                    rows.append({
                        "v0": params["v0"],
                        "kappa": params["kappa"],
                        "theta": params["theta"],
                        "sigma_v": params["sigma_v"],
                        "rho": params["rho"],
                        "K": K,
                        "T": T,
                        "implied_vol": implied_vol,
                    })

    return rows


training_rows = build_training_rows(
    parameter_sets=parameter_sets,
    strike_grid=STRIKE_GRID,
    maturity_grid=MATURITY_GRID,
    S0=S0,
    r=RISK_FREE_RATE,
    q=DIVIDEND_YIELD,
)

training_rows[:5]


[{'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'K': 85.0,
  'T': 0.019178082191780823,
  'implied_vol': 0.29198117486892017},
 {'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'K': 90.0,
  'T': 0.019178082191780823,
  'implied_vol': 0.2865504868879416},
 {'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'K': 95.0,
  'T': 0.019178082191780823,
  'implied_vol': 0.28176521306667657},
 {'v0': 0.07499616703223104,
  'kappa': 2.1532996637634767,
  'theta': 0.18778262433281417,
  'sigma_v': 0.2971916669149291,
  'rho': -0.2730263123203287,
  'K': 100.0,
  'T': 0.019178082191780823,
  'implied_vol': 0.27777128824342007},
 {'v0': 0.07499616703223104,
  'kappa': 2.15329966376347

## 3. Save the Training Table

The saved CSV contains only the feature columns and target implied volatility needed for GP training.

In [6]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

fieldnames = ["v0", "kappa", "theta", "sigma_v", "rho", "K", "T", "implied_vol"]
with OUTPUT_PATH.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(training_rows)

implied_vols = np.array([row["implied_vol"] for row in training_rows])

print(f"Saved {len(training_rows):,} rows to {OUTPUT_PATH}")
print(f"Parameter sets: {len(parameter_sets):,}")
print(f"Grid size per set: {len(STRIKE_GRID)} strikes x {len(MATURITY_GRID)} maturities")
print(f"Implied vol range: {implied_vols.min():.4f} to {implied_vols.max():.4f}")


Saved 51,867 rows to data/simulated_training_data.csv
Parameter sets: 1,500
Grid size per set: 7 strikes x 5 maturities
Implied vol range: 0.0538 to 0.5466
